# Huấn luyện nhận diện cảm xúc trên Google Colab
Notebook lưu checkpoint trên Google Drive và cho phép tải model về máy local. Trước khi chạy, chọn **Runtime → Change runtime type → T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/facial-emotion-recognition')
DATA_ZIP = PROJECT_DIR / 'fer2013.zip'  # đặt None nếu data/ đã được giải nén
EPOCHS = 30
BATCH_SIZE = 128
PATIENCE = 7
RESUME = False
MODEL_PATH = PROJECT_DIR / 'models/emotion_resnet18.pt'
assert PROJECT_DIR.is_dir(), f'Không tìm thấy {PROJECT_DIR}'
print('Project:', PROJECT_DIR)

In [ ]:
import os, sys, torch
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Chưa bật GPU: Runtime → Change runtime type → T4 GPU')

In [ ]:
import zipfile
data_dir = PROJECT_DIR / 'data'
if DATA_ZIP is not None and Path(DATA_ZIP).is_file() and not (data_dir / 'train').is_dir():
    print('Đang giải nén', DATA_ZIP)
    data_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as archive:
        archive.extractall(data_dir)

train_dir = data_dir / 'train'
val_dir = data_dir / 'val' if (data_dir / 'val').is_dir() else data_dir / 'test'
assert train_dir.is_dir(), f'Thiếu {train_dir}'
assert val_dir.is_dir(), f'Thiếu {val_dir}'
for split in (train_dir, val_dir):
    counts = {p.name: sum(1 for f in p.rglob('*') if f.is_file()) for p in split.iterdir() if p.is_dir()}
    print(split.name, counts)
assert sorted(p.name for p in train_dir.iterdir() if p.is_dir()) == sorted(p.name for p in val_dir.iterdir() if p.is_dir()), 'Nhãn train/val không khớp'

In [ ]:
# Colab thường đã có torchvision; cài bổ sung nếu runtime chưa có.
import subprocess
try:
    import torchvision
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchvision'])
    import torchvision
print('torchvision:', torchvision.__version__)


In [ ]:
# Luôn chạy mã nguồn trong PROJECT_DIR trên Drive. Không clone repo khác,
# nếu không train.py có thể là phiên bản cũ không chứa ResNet-18.
os.chdir(PROJECT_DIR)
print('Training code:', Path.cwd())

In [ ]:
from pathlib import Path
import shutil
import zipfile

DRIVE_DIR = Path(
    "/content/drive/MyDrive/facial-emotion-recognition"
)

LOCAL_DIR = Path("/content/emotion-data")
LOCAL_ZIP = Path("/content/fer2013.zip")

shutil.copy2(DRIVE_DIR / "fer2013.zip", LOCAL_ZIP)

LOCAL_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LOCAL_ZIP) as archive:
    archive.extractall(LOCAL_DIR)

print("Dataset local:", LOCAL_DIR)

In [ ]:
TRAIN_DIR = "/content/emotion-data/train"
VAL_DIR = "/content/emotion-data/test"

MODEL_PATH = (
    "/content/drive/MyDrive/"
    "facial-emotion-recognition/models/emotion_resnet18.pt"
)

In [ ]:
!python train.py \
  --train-dir "$TRAIN_DIR" \
  --val-dir "$VAL_DIR" \
  --output "$MODEL_PATH" \
  --epochs 100 \
  --batch-size 256 \
  --learning-rate 0.0003 \
  --input-size 96 \
  --pretrained \
  --patience 15 \
  --num-workers 4 \
  --device cuda \
  --amp

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
print({k: checkpoint[k] for k in ('architecture', 'labels', 'epoch', 'val_accuracy', 'input_size')})
print(f'Model: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1024 / 1024:.1f} MB)')

from google.colab import files
files.download(str(MODEL_PATH))